In [7]:

import win32com.client
import csv

NETWORK = r"D:\BTP_Vissim\simulation.inpx"

# Actual VISSIM lane identifiers
APPROACH_LANES = {"1-1", "1-2"}

SIMULATION_STEPS = 3900

vissim = win32com.client.DispatchEx("Vissim.Vissim")
vissim.LoadNet(NETWORK)

# Disable visualization
vissim.Graphics.CurrentNetworkWindow.SetAttValue("QuickMode", 1)

seen_vehicles = set()
vehicle_count = 0

# Cache vehicle dimensions
dimensions = {}

# Store all detected vehicle records
vehicle_data = []

for step in range(SIMULATION_STEPS):

    vissim.Simulation.RunSingleStep()

    sim_time = step + 1

    # Get current vehicles
    vehicles = vissim.Net.Vehicles.GetAll()

    for i in range(len(vehicles)):

        vehicle = vehicles[i]

        vehicle_id = int(vehicle.AttValue("No"))

        # Get actual lane identifier, e.g. 1-1 or 1-2
        lane_id = str(vehicle.AttValue("Lane")).strip()

        # Select only Link 1, Lane 1 and Lane 2
        if lane_id not in APPROACH_LANES:
            continue

        # Count each vehicle only once
        if vehicle_id in seen_vehicles:
            continue

        seen_vehicles.add(vehicle_id)
        vehicle_count += 1

        # Vehicle type
        vehicle_type_id = int(vehicle.AttValue("VehType"))

        # Get vehicle dimensions
        if vehicle_type_id not in dimensions:

            vehicle_type = vissim.Net.VehicleTypes.ItemByKey(
                vehicle_type_id
            )

            dimensions[vehicle_type_id] = {
                "length" : float(vehicle.AttValue("Length")),
                "width" : float(vehicle.AttValue("Width"))
            }

        dims = dimensions[vehicle_type_id]

        # Create vehicle record
        record = {
            "time": sim_time,
            "vehicle_id": vehicle_id,
            "lane": lane_id,
            "vehicle_type_id": vehicle_type_id,
            "length_m": dims["length"],
            "width_m": dims["width"],
            "total_vehicles_seen": vehicle_count
        }

        vehicle_data.append(record)

        print(record)

# Final count
print("\nTotal vehicles detected:", vehicle_count)


{'time': 21, 'vehicle_id': 1, 'lane': '1-1', 'vehicle_type_id': 610, 'length_m': 1.7747000455856323, 'width_m': 0.6274850070476532, 'total_vehicles_seen': 1}
{'time': 53, 'vehicle_id': 2, 'lane': '1-1', 'vehicle_type_id': 610, 'length_m': 1.7747000455856323, 'width_m': 0.6274850070476532, 'total_vehicles_seen': 2}
{'time': 66, 'vehicle_id': 3, 'lane': '1-2', 'vehicle_type_id': 610, 'length_m': 1.7747000455856323, 'width_m': 0.6274850070476532, 'total_vehicles_seen': 3}
{'time': 77, 'vehicle_id': 4, 'lane': '1-2', 'vehicle_type_id': 190, 'length_m': 6.041460394859314, 'width_m': 2.396122297759108, 'total_vehicles_seen': 4}
{'time': 107, 'vehicle_id': 5, 'lane': '1-2', 'vehicle_type_id': 100, 'length_m': 4.414452478289604, 'width_m': 2.047415933872572, 'total_vehicles_seen': 5}
{'time': 160, 'vehicle_id': 6, 'lane': '1-1', 'vehicle_type_id': 610, 'length_m': 1.7747000455856323, 'width_m': 0.6274850070476532, 'total_vehicles_seen': 6}
{'time': 174, 'vehicle_id': 7, 'lane': '1-2', 'vehicle

com_error: (-2147418111, 'Call was rejected by callee.', None, None)

In [ ]:

# Export results to CSV
with open("VISSIM_Vehicle_Data.csv", "w", newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "time",
            "vehicle_id",
            "lane",
            "vehicle_type_id",
            "length_m",
            "width_m",
            "total_vehicles_seen"
        ]
    )

    writer.writeheader()
    writer.writerows(vehicle_data)

print("\nData successfully saved to VISSIM_Vehicle_Data.csv")
